<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week13_Transformer/Transformer_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Transformer-like (both encoder and decoder) model from scratch for Text Generation

with customized attention block

At the end of this session, you will be able to:
- prepare data and use model-specific Tokenizer to format data suitable for use by the model
- construct the transformer model 
- train the model for binary and multi-class text classification


### Install Hugging Face Transformers library

If you are running this notebook in Google Colab, you will need to install the Hugging Face transformers library as it is not part of the standard environment.

In [1]:
!pip install datasets

In [2]:
!pip install transformers

In [3]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Find a dataset

First, let us find a corpus of text in Esperanto. Here we’ll use the Esperanto portion of the [OSCAR corpus](https://traces1.inria.fr/oscar/) from INRIA.
OSCAR is a huge multilingual corpus obtained by language classification and filtering of [Common Crawl](https://commoncrawl.org/) dumps of the Web.

<img src="https://huggingface.co/blog/assets/01_how-to-train/oscar.png" style="margin: auto; display: block; width: 260px;">

The Esperanto portion of the dataset is only 299M, so we’ll concatenate with the Esperanto sub-corpus of the [Leipzig Corpora Collection](https://wortschatz.uni-leipzig.de/en/download), which is comprised of text from diverse sources like news, literature, and wikipedia.

The final training corpus has a size of 3 GB, which is still small – for your model, you will get better results the more data you can get to pretrain on. 



In [4]:
# # in this notebook we'll only get one of the files (the Oscar one) for the sake of simplicity and performance
# !wget -c https://cdn-datasets.huggingface.co/EsperBERTo/data/oscar.eo.txt

## 2. Train a tokenizer

We choose to train a byte-level Byte-pair encoding tokenizer (the same as GPT-2), with the same special tokens as RoBERTa. Let’s arbitrarily pick its size to be 52,000.

We recommend training a byte-level BPE (rather than let’s say, a WordPiece tokenizer like BERT) because it will start building its vocabulary from an alphabet of single bytes, so all words will be decomposable into tokens (no more `<unk>` tokens!).

In [5]:
from pathlib import Path
paths = [str(x) for x in Path(".").glob("**/*.txt")]

In [6]:
%%time 
from tokenizers import ByteLevelBPETokenizer
vocab_size = 52000

# Initialize a tokenizer
tokenizer = ByteLevelBPETokenizer() # BPE: Byte Pair Encoding, a subword tokenization algorithm

# Customize training
tokenizer.train(files=paths, vocab_size=vocab_size, min_frequency=2, special_tokens=[
    "<s>",
    "<pad>",
    "</s>",
    "<unk>",
    "<mask>",
])

CPU times: total: 4min 41s
Wall time: 33.3 s


In [7]:
tokenizer.get_vocab_size()

52000

In [8]:
tokenizer.encode("The Eiffel Tower is located in").ids

[7691, 44720, 39078, 2199, 558, 621, 7547, 326]

In [ ]:
# tokenizer.save("tokenizer-BPE/tokenizer.json")

import os
os.mkdir('tokenizer-BPE')
tokenizer_dir = 'tokenizer-BPE'
tokenizer_file = os.path.join(tokenizer_dir, 'tokenizer.json')
tokenizer.save(tokenizer_file)

In [10]:
import json

config = {
    "unk_token": "<unk>",
    "pad_token": "<pad>",
    "cls_token": "<cls>",
    "sep_token": "<sep>",
    "mask_token": "<mask>",
    "bos_token": "<s>",
    "eos_token": "</s>",
    "model_max_length": 128,
    "tokenizer_class": "PreTrainedTokenizerFast"
}
with open("tokenizer-BPE/tokenizer_config.json", "w") as f:
    json.dump(config, f)

# 3. Data Pre-processing an Dataloader Construction

In [11]:
from transformers import PreTrainedTokenizerFast

fast_tok = PreTrainedTokenizerFast.from_pretrained(
    "./tokenizer-BPE",
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<cls>",
    sep_token="<sep>",
    mask_token="<mask>",
)


c:\Users\liangnanyi\.conda\envs\it3103env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from datasets import load_dataset

# Load the text file as a dataset
dataset = load_dataset("text", data_files={"train": "./oscar.eo.txt"})

# Tokenize the dataset
def tokenize_function(examples):
    return fast_tok(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [13]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Assume tokenized_datasets["train"]["input_ids"] exists
input_ids = np.array(tokenized_datasets["train"]["input_ids"])

# Split into train and validation (90% train, 10% val)
train_ids, val_ids = train_test_split(input_ids, test_size=0.2, random_state=42)
train_ids = train_ids.astype(np.int32)
val_ids = val_ids.astype(np.int32)

def create_tf_dataset(input_ids, batch_size=32, seq_len=128):
    # Remove sequences shorter than seq_len
    input_ids = input_ids[[len(seq) == seq_len for seq in input_ids]]
    ds = tf.data.Dataset.from_tensor_slices(input_ids)
    ds = ds.batch(batch_size, drop_remainder=True)
    for batch in ds:
        batch_src = batch[:, :-1]
        batch_tgt = batch[:, 1:]
        yield batch_src, batch_tgt

batch_size = 32
seq_len = 128

data_loader = create_tf_dataset(train_ids, batch_size, seq_len)
val_data_loader = create_tf_dataset(val_ids, batch_size, seq_len)

## 4. Construct a customized Transformer-like model

Now let us fine-tune the pre-trained model by training it with our custom dataset.  

We will instantiate a pretrained model 'distilbert-base-uncased', using `TFAutoModelForSequenceClassification`, and passing `num_labels=2` to indicate we want to train a 2-class (binary) classifier.

The model is a `tf.keras.Model` subclass. So you can train the model using Keras API such as `fit()`.

In [14]:
import tensorflow as tf


class CustomMHA(tf.keras.layers.Layer):
    """
    Multi-head attention with
      • Rotary positional embedding
      • Per-head learnable gate alpha_h
      • Query/Key dropout (before similarity)
    Supports:
      mode='self'   - encoder or causal decoder self-attention
      mode='cross'  - decoder --> encoder cross attention
    """

    def __init__(self, hidden, heads, qk_drop=0.1, attn_drop=0.1, **kw):
        super().__init__(**kw)
        assert hidden % heads == 0
        self.h, self.n = hidden, heads          # hidden, #heads
        self.dk = hidden // heads
        self.qk_drop = qk_drop
        self.attn_drop = attn_drop

        self.to_q = tf.keras.layers.Dense(hidden, use_bias=False)
        self.to_k = tf.keras.layers.Dense(hidden, use_bias=False)
        self.to_v = tf.keras.layers.Dense(hidden, use_bias=False)
        self.proj = tf.keras.layers.Dense(hidden)

        self.alpha = self.add_weight("alpha", shape=(heads,),
                                     initializer="ones")

    # ---- rotary helper ---- #
    def _rope(self, t):
        # t: (B, H, L, dk)   dk even
        dk2 = self.dk // 2
        freq = tf.range(dk2, dtype=t.dtype) / dk2
        freq = 1. / (10000. ** freq)
        pos  = tf.range(tf.shape(t)[-2], dtype=t.dtype)
        angle = tf.einsum("l,f->lf", pos, freq)    # (L,dk/2)
        cos, sin = tf.cos(angle), tf.sin(angle)
        t1, t2 = tf.split(t, 2, axis=-1)
        return tf.concat([t1*cos - t2*sin, t1*sin + t2*cos], axis=-1)

    def _split(self, z):
        # (B,L,H) ➜ (B,n,L,dk)
        B = tf.shape(z)[0]
        z = tf.reshape(z, [B, -1, self.n, self.dk])
        return tf.transpose(z, [0, 2, 1, 3])

    # ---- forward ---- #
    def call(self, q_inp, k_inp=None, v_inp=None,
             mask=None, causal=False, training=False):
        if k_inp is None:            # self-attention
            k_inp = v_inp = q_inp

        q = self._split(self.to_q(q_inp))
        k = self._split(self.to_k(k_inp))
        v = self._split(self.to_v(v_inp))

        if k_inp is q_inp:           # only self-attn gets RoPE
            q, k = map(self._rope, (q, k))

        # — Query/Key dropout —
        if training and self.qk_drop:
            drop_mask = tf.random.uniform(tf.shape(q)) >= self.qk_drop
            q = tf.where(drop_mask, q, tf.zeros_like(q))
            drop_mask = tf.random.uniform(tf.shape(k)) >= self.qk_drop
            k = tf.where(drop_mask, k, tf.zeros_like(k))

        logits = tf.matmul(q, k, transpose_b=True)      # (B,n,Lq,Lk)
        logits /= tf.math.sqrt(tf.cast(self.dk, logits.dtype))
        logits *= self.alpha[None, :, None, None]       # head gate

        # causal mask if needed
        if causal:
            Lq, Lk = tf.shape(q)[-2], tf.shape(k)[-2]
            mask_tri = tf.linalg.band_part(tf.ones((Lq, Lk), logits.dtype), -1, 0)
            logits += (1.0 - mask_tri) * -1e9

        # padding mask
        if mask is not None:         # mask: (B,1,1,Lk) bool
            mask = tf.cast(mask, tf.bool)
            logits = tf.where(mask, logits, tf.constant(-1e9, logits.dtype))

        attn = tf.nn.softmax(logits, axis=-1)
        if training and self.attn_drop:
            attn = tf.nn.dropout(attn, self.attn_drop)

        out = tf.matmul(attn, v)                            # (B,n,Lq,dk)
        out = tf.transpose(out, [0, 2, 1, 3])               # (B,Lq,n,dk)
        out = tf.reshape(out, [tf.shape(out)[0], -1, self.h])   # (B,Lq,h)
        return self.proj(out)


In [15]:
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(self, h, n, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.mha   = CustomMHA(h, n, attn_drop=drop)
        self.mlp   = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(h*mlp_ratio, activation="gelu"),
                tf.keras.layers.Dropout(drop),
                tf.keras.layers.Dense(h),
                tf.keras.layers.Dropout(drop),
            ]
        )

    def call(self, x, mask=None, training=False):
        x = x + self.mha(self.norm1(x), mask=mask, training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        return x


class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, h, n, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.norm3 = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.self_mha  = CustomMHA(h, n, attn_drop=drop)
        self.cross_mha = CustomMHA(h, n, attn_drop=drop)
        self.mlp = tf.keras.Sequential(
            [
                tf.keras.layers.Dense(h*mlp_ratio, activation="gelu"),
                tf.keras.layers.Dropout(drop),
                tf.keras.layers.Dense(h),
                tf.keras.layers.Dropout(drop),
            ]
        )

    def call(self, y, enc, self_mask=None, enc_mask=None, training=False):
        y = y + self.self_mha(
            self.norm1(y), causal=True, mask=self_mask, training=training
        )
        y = y + self.cross_mha(
            self.norm2(y), k_inp=enc, v_inp=enc,
            mask=enc_mask, training=training
        )
        y = y + self.mlp(self.norm3(y), training=training)
        return y


In [16]:
class Transformer(tf.keras.Model):
    """
    Encoder-Decoder with shared token embeddings (tied softmax).
    """

    def __init__(
        self,
        vocab,
        max_len,
        n_enc=6,
        n_dec=6,
        hidden=512,
        heads=8,
        mlp_ratio=4,
        drop=0.1,
        **kw,
    ):
        super().__init__(**kw)
        self.vocab = vocab
        self.hidden = hidden
        self.max_len = max_len

        self.emb = tf.keras.layers.Embedding(vocab, hidden)
        self.pos = self.add_weight("pos_emb", shape=(max_len, hidden),
                                   initializer="zeros")
        self.drop = tf.keras.layers.Dropout(drop)

        self.enc_layers = [EncoderBlock(hidden, heads, mlp_ratio, drop)
                           for _ in range(n_enc)]
        self.dec_layers = [DecoderBlock(hidden, heads, mlp_ratio, drop)
                           for _ in range(n_dec)]

        self.enc_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)
        self.dec_norm = tf.keras.layers.LayerNormalization(epsilon=1e-5)

        # tie softmax with token embedding
        self.lm_head = tf.keras.layers.Dense(
            vocab, use_bias=False,
            kernel_initializer="zeros"
        )

    # ------------------- forward (teacher-forcing) ------------------- #
    def call(self, inputs, training=False):
        """
        src : (B, Ls)  input IDs  (0 = pad)
        tgt : (B, Lt)  target IDs shifted right  (BOS token first)
        """
        src, tgt = inputs
        src_mask = tf.not_equal(src, 0)[:, None, None, :]  # (B,1,1,Ls)
        enc = self._encode(src, src_mask, training)

        self_mask = tf.not_equal(tgt, 0)[:, None, None, :]  # padding mask
        # causal portion added inside decoder blocks

        dec = self._decode(tgt, enc, self_mask, src_mask, training)
        logits = self.lm_head(dec) * (self.hidden ** -0.5)  # weight tying scale
        return logits

    # ------------------- encoder helper ------------------- #
    def _encode(self, src, src_mask, training):
        x = self.emb(src) + self.pos[: tf.shape(src)[1]]
        x = self.drop(x, training=training)
        for blk in self.enc_layers:
            x = blk(x, mask=src_mask, training=training)
        return self.enc_norm(x)

    # ------------------- decoder helper ------------------- #
    def _decode(self, tgt, enc, self_mask, enc_mask, training):
        y = self.emb(tgt) + self.pos[: tf.shape(tgt)[1]]
        y = self.drop(y, training=training)
        for blk in self.dec_layers:
            y = blk(y, enc, self_mask, enc_mask, training=training)
        return self.dec_norm(y)

    # ------------------- greedy / beam search ------------------- #
    @tf.function(
        input_signature=[tf.TensorSpec(shape=[None, None], dtype=tf.int32),
                         tf.TensorSpec(shape=(), dtype=tf.int32)])
    def generate(self, src, max_len=64, beam=1):
        """
        Greedy if beam==1 else beam search.  Returns (B, max_len) IDs.
        BOS token (e.g. 1) is appended automatically.

        *No caching* for simplicity fine for demo / <1k tokens.
        """
        B = tf.shape(src)[0]
        bos = tf.ones([B, 1], tf.int32)   # assume ID 1 is <bos>
        seqs = bos                       # growing tensor

        # pre-compute encoder memory once
        src_mask = tf.not_equal(src, 0)[:, None, None, :]
        enc = self._encode(src, src_mask, training=False)

        if beam > 1:
            raise NotImplementedError("Beam search omitted for brevity.")

        for _ in tf.range(max_len):
            logits = self._decode(
                seqs, enc,
                self_mask=tf.ones_like(seqs)[:, None, None, :],
                enc_mask=src_mask,
                training=False
            )
            next_token_logits = self.lm_head(logits[:, -1]) * (self.hidden**-0.5)
            next_ids = tf.argmax(next_token_logits, axis=-1, output_type=tf.int32)
            seqs = tf.concat([seqs, next_ids[:, None]], axis=1)

            # stop if all ended with <eos> (assume id 2)
            if tf.reduce_all(tf.equal(next_ids, 2)):
                break
        return seqs[:, 1:]               # strip BOS


In [17]:
MAX_LEN    = 128

model = Transformer(
    vocab=vocab_size, max_len=MAX_LEN,
    n_enc=6, n_dec=6, hidden=512, heads=8
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)

def loss_with_mask(y_true, y_pred):
    # y_true: (B,Lt) next tokens
    mask = tf.cast(tf.not_equal(y_true, 0), y_pred.dtype)
    loss = loss_fn(y_true, y_pred)
    return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(3e-4, weight_decay=1e-2),
    loss=loss_with_mask
)

# model.summary(line_length=130)



In [18]:
def compute_accuracy(y_true, y_pred):
    y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    y_true = tf.cast(y_true, tf.int32)
    mask = tf.not_equal(y_true, 0)
    correct = tf.reduce_sum(tf.cast(tf.equal(y_true, y_pred_ids) & mask, tf.float32))
    total = tf.reduce_sum(tf.cast(mask, tf.float32))
    return correct / total if total > 0 else 0.0

In [ ]:
import time

epochs = 10
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    batch_num = 0
    total_loss = 0.0
    total_acc = 0.0
    start_time = time.time()
    for batch_src, batch_tgt in data_loader:      # (B,Ls), (B,Lt)
        inp = batch_tgt[:, :-1]
        lbl = batch_tgt[:, 1:]
        preds = model([batch_src, inp], training=True)
        loss = model.train_on_batch([batch_src, inp], lbl)
        acc = compute_accuracy(lbl, preds)
        total_loss += loss
        total_acc += acc
        batch_num += 1
        if batch_num % 2 == 0:
            avg_loss = total_loss / batch_num
            avg_acc = total_acc / batch_num
            elapsed = time.time() - start_time
            print(f"  Batch {batch_num}: Loss={loss:.4f}, Acc={acc:.4f}, Avg Loss={avg_loss:.4f}, Avg Acc={avg_acc:.4f}, Elapsed={elapsed:.1f}s")
    avg_loss = total_loss / batch_num if batch_num > 0 else 0
    avg_acc = total_acc / batch_num if batch_num > 0 else 0
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1} finished: Avg Loss={avg_loss:.4f}, Avg Acc={avg_acc:.4f}, Time={epoch_time:.1f}s, Batches={batch_num}")

    # Validation loop (define val_data_loader similarly to data_loader)
    val_loss = 0.0
    val_acc = 0.0
    val_batches = 0
    # Uncomment and define val_data_loader for validation
    for val_src, val_tgt in val_data_loader:
        val_inp = val_tgt[:, :-1]
        val_lbl = val_tgt[:, 1:]
        val_preds = model([val_src, val_inp], training=False)
        v_loss = loss_with_mask(val_lbl, val_preds).numpy()
        v_acc = compute_accuracy(val_lbl, val_preds)
        val_loss += v_loss
        val_acc += v_acc
        val_batches += 1
    if val_batches > 0:
        print(f"  Validation: Loss={val_loss/val_batches:.4f}, Acc={val_acc/val_batches:.4f}")

Epoch 1/10
  Batch 2: Loss=10.6032, Acc=0.4903, Avg Loss=10.6017, Avg Acc=0.5226, Elapsed=33.1s
  Batch 4: Loss=10.5760, Acc=0.5645, Avg Loss=10.5891, Avg Acc=0.5405, Elapsed=66.3s
  Batch 6: Loss=10.5713, Acc=0.4985, Avg Loss=10.5851, Avg Acc=0.5228, Elapsed=101.4s


In [ ]:
test_src = tokenizer.encode("The Eiffel Tower is located in").ids
test_src = np.array(test_src)[None, :]  # shape (1, seq_len)
generated = model.generate(tf.constant(test_src, dtype=tf.int32), max_len=40)
print("▶", tokenizer.decode(generated[0].numpy()))


ValueError: in user code:

    File "C:\Users\liangnanyi\AppData\Local\Temp\ipykernel_29936\3010083027.py", line 97, in generate  *
        for _ in tf.range(max_len):

    ValueError: 'seqs' has shape (None, 1) before the loop, but shape (None, 2) after one iteration. Use tf.autograph.experimental.set_loop_options to set shape invariants.


In [ ]:
# from transformers import TFAutoModelForSequenceClassification

# model = TFAutoModelForSequenceClassification.from_pretrained(
#         "distilbert-base-uncased",num_labels=2)

Transformer models benefit from a much lower learning rate than the default used by Adam, which is 1e-3. In this training, we will start the training with 5e-5 (0.00005) and slowly reduce the learning rate over the course of training. In the literature, you will sometimes see this referred to as decaying or annealing the learning rate. In Keras, the best way to do this is to use a learning rate scheduler. A good one to use is PolynomialDecay. Despite the name, with default settings it simply linearly decays the learning rate from the initial value to the final value over the course of training, which is exactly what we want. In order to use a scheduler correctly, though, we need to tell it how long training is going to be. We compute that as `num_train_steps` below.

In [ ]:
from tensorflow.keras.optimizers.schedules import PolynomialDecay

num_epochs = 10

# The number of training steps is the number of samples in the dataset, divided by the batch size then multiplied
# by the total number of epochs. Since our dataset is already batched, we can simply take the len.
num_train_steps = len(train_dataset) * num_epochs

lr_scheduler = PolynomialDecay(
    initial_learning_rate=5e-5, end_learning_rate=0.0, decay_steps=num_train_steps
)

Now we will just compile the model with the learning rate scheduler and the loss function and train our model for 1 epoch. 

Note that the transformer model output logits directly instead of going through a softmax layer. In your loss function, you will need to set `from_logits=True`.


In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

opt = Adam(learning_rate=lr_scheduler)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])

model.fit(train_dataset, validation_data=val_dataset, epochs=num_epochs)

You will notice that validation accuracy reaches around 89%.  Let's evaluate on our test set. We should see around the same accuracy. 


In [ ]:
model.evaluate(test_dataset)

Let's just go ahead and save our model for inference later. Note that we use transformers library specific save method `save_pretrained()` instead of normal keras model save.

In [ ]:
model.save_pretrained('finetuned_model')

## Try out the model

Now let's try out our model with our own sentence.  We first load our saved fined-tuned model using `from_pretrained()` method and specify the folder name where we saved the model to.

In [ ]:
my_model = TFAutoModelForSequenceClassification.from_pretrained(
        "finetuned_model")

In [ ]:
text = input('Write your review here:')

In [ ]:
inputs = tokenizer(text, return_tensors="tf")
output = my_model(inputs)
pred_prob = tf.nn.softmax(output.logits, axis=-1)
print(pred_prob)
pred = np.argmax(pred_prob)
print(pred)
if pred == 1:
    print('positive')
else:
    print('negative')

**Exercise:**

Now, try to fine-tune DistilBERT for  multi-class text classification task using this [dataset](https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/news.csv) that groups news title into 4 categories: e (entertainment), b (business), t (tech), m (medical/health). Original dataset can be found [here](https://archive.ics.uci.edu/ml/datasets/News+Aggregator)

*Hint*:

- The csv file is using tab as delimiter, so you need to specify `delimiter='\t'` when you use `pd.read_csv()`
- You should also write a separate function to map the 4 character labels `('e','t','b','m')` into its numeric labels
- Remember to change the `num_labels` to the appropriate number when you instantiate the DistilBert SequenceClassification model.